# ДЗ 4. Задание 2

In [32]:
from Bio import SeqIO
from Bio import motifs
from Bio.Seq import Seq
import pandas as pd

In [33]:
records = SeqIO.parse("chr1.fna", "fasta")
record = next(records)
sequence_all = record.seq
sequence = sequence_all[:1000000]

In [18]:
sites = ["GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA", "ACAGTCAGC", "TAGGTCAGC",
         "CAGGTCAGC", "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC", "CAGGTTGGC"]

ps = [Seq(site) for site in sites]

m = motifs.create(ps)
m

In [21]:
pfm = m.counts.normalize(pseudocounts=0.1)

In [26]:
df = pd.DataFrame(pfm).T
df

,0,1,2,3,4,5,6,7,8
A,0.105769,0.778846,0.105769,0.009615,0.009615,0.201923,0.682692,0.201923,0.105769
C,0.586538,0.201923,0.105769,0.009615,0.009615,0.586538,0.009615,0.009615,0.778846
G,0.105769,0.009615,0.778846,0.971154,0.009615,0.009615,0.298077,0.778846,0.009615
T,0.201923,0.009615,0.009615,0.009615,0.971154,0.201923,0.009615,0.009615,0.105769


совпадает c PPM из предыдущего задания

In [30]:
background = {'A':0.295,'T':0.295,'G':0.205,'C':0.205}
pwm = pfm.log_odds(background)

In [31]:
df = pd.DataFrame(pwm).T
df

,0,1,2,3,4,5,6,7,8
A,-1.479795,1.400623,-1.479795,-4.939227,-4.939227,-0.546909,1.210521,-0.546909,-1.479795
C,1.516602,-0.021818,-0.954704,-4.414136,-4.414136,1.516602,-4.414136,-4.414136,1.925714
G,-0.954704,-4.414136,1.925714,2.244076,-4.414136,-4.414136,0.540061,1.925714,-4.414136
T,-0.546909,-4.939227,-4.939227,-4.939227,1.718985,-0.546909,-4.939227,-4.939227,-1.479795


совпадает с PWM из предыдущего задания

In [19]:
m.consensus

Seq('CAGGTCAGC')

Это тоже совпало с последовательностью максимаьного скора в предыдущем задании, ура :)

pwm.search дает в обратной цепи отрицательные позиции, при этом реальная позиция будет равна длина сиквенса + отрицательная позиция:

In [41]:
hits = []

for position, score in pwm.search(sequence, threshold=5.0):
    if position >= 0:
        hits.append((position, "прямая  ", score))
    else:
        true_position = len(sequence) + position
        hits.append((true_position, "обратная", score))

In [43]:
hits.sort(key=lambda x: x[0])
with open("HW_4_2.txt", "w") as f:
    f.write("позиция\tцепь\t\tскор\n")
    for pos, strand, score in hits:
        f.write(f"{pos}\t{strand}\t{score:.3f}\n")